<a href="https://colab.research.google.com/github/Mahefa-MaH/ml_plantvision_efficientnet_v2/blob/main/M2S10_de_kaggle_colab_efficientNet_plante_disease_class_multiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Install necessary libraries
!pip install -q kaggle efficientnet_pytorch mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

### Dataset Download and Extraction (New Plant Diseases)

Similarly, this section downloads another plant disease dataset from Kaggle, specifically a 'new plant diseases dataset'. It then extracts the downloaded ZIP file into a designated directory. This dataset will be used for training and validating a plant disease classification model.

In [2]:
import kagglehub
import os
import shutil

# Ensure the target base directory exists for our restructured data
# This also creates the intermediate 'npd' directory.
!mkdir -p kaggle_data/new-plant-diseases_dataset/npd/npd-train

# Download the dataset using kagglehub.dataset_download
# This function returns the path to the extracted dataset files.
NEW_PLANT_DISEASES_KAGGLEHUB_ROOT = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')

print(f"KaggleHub dataset downloaded to: {NEW_PLANT_DISEASES_KAGGLEHUB_ROOT}")

# Define paths based on the structure returned by kagglehub and the desired target structure
# The 'new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)' is the actual root for train/valid
original_train_valid_source = os.path.join(NEW_PLANT_DISEASES_KAGGLEHUB_ROOT, 'new plant diseases dataset(augmented)', 'New Plant Diseases Dataset(Augmented)')
original_test_source = os.path.join(NEW_PLANT_DISEASES_KAGGLEHUB_ROOT, 'test')

# Correct target paths according to the plan
target_npd_train_base = 'kaggle_data/new-plant-diseases_dataset/npd/npd-train'
target_test_base = 'kaggle_data/new-plant-diseases_dataset/test'

# Create target directories if they don't exist
os.makedirs(os.path.join(target_npd_train_base, 'train'), exist_ok=True)
os.makedirs(os.path.join(target_npd_train_base, 'valid'), exist_ok=True)
os.makedirs(target_test_base, exist_ok=True)

# Copy train and valid directories instead of moving, due to cross-device link issue and read-only source
shutil.copytree(os.path.join(original_train_valid_source, 'train'), os.path.join(target_npd_train_base, 'train'), dirs_exist_ok=True)
shutil.copytree(os.path.join(original_train_valid_source, 'valid'), os.path.join(target_npd_train_base, 'valid'), dirs_exist_ok=True)

# Copy the test directory
shutil.copytree(original_test_source, target_test_base, dirs_exist_ok=True)

print("Dataset restructured successfully.")

# Clean up the intermediate kagglehub download directory if it's no longer needed
# Note: This is commented out as the source is a read-only file system.
# shutil.rmtree(NEW_PLANT_DISEASES_KAGGLEHUB_ROOT)

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
KaggleHub dataset downloaded to: /kaggle/input/new-plant-diseases-dataset
Dataset restructured successfully.


### Install EfficientNet PyTorch Library

This cell installs the `efficientnet_pytorch` library, which provides pre-trained EfficientNet models for PyTorch. EfficientNet is a family of convolutional neural networks that achieve state-of-the-art accuracy with fewer parameters and FLOPs than previous models, making them suitable for image classification tasks like plant disease detection.

In [3]:
!pip install efficientnet_pytorch

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=bfaa03885cf8147c424b0b4d05f308602e1aca7e38aa908d8cbf46c47b9f25bf
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet_pytorch


### Data Inspection and Preparation

This section verifies the structure and content of the downloaded datasets. It prints the class names available in the training and validation sets, as well as the files in the test directory. It also checks for the file extensions to ensure data consistency and removes any `.ipynb_checkpoints` directories that might interfere with data loading.

In [4]:
#!ls kaggle_data/rice_dataset/images/train

In [5]:
# !mkdir kaggle_data/new-plant-diseases_dataset/npd/npd/train/temp
# !mkdir kaggle_data/new-plant-diseases_dataset/npd/npd/test/temp

In [6]:
import os
print("Classes dans train :", os.listdir('kaggle_data/new-plant-diseases_dataset/npd/npd-train/train'))
print("Classes dans valid :", os.listdir('kaggle_data/new-plant-diseases_dataset/npd/npd-train/valid'))
print("Fichiers dans test :", os.listdir('kaggle_data/new-plant-diseases_dataset/test/test'))

Classes dans train : ['Corn_(maize)___Common_rust_', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Northern_Leaf_Blight', 'Apple___healthy', 'Peach___Bacterial_spot', 'Cherry_(including_sour)___healthy', 'Tomato___Leaf_Mold', 'Tomato___Early_blight', 'Potato___Late_blight', 'Peach___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Grape___Esca_(Black_Measles)', 'Potato___healthy', 'Tomato___Tomato_mosaic_virus', 'Strawberry___Leaf_scorch', 'Tomato___Septoria_leaf_spot', 'Tomato___healthy', 'Raspberry___healthy', 'Squash___Powdery_mildew', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Soybean___healthy', 'Apple___Cedar_apple_rust', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Apple___Black_rot', 'Strawberry___healthy', 'Tomato___Target_Spot', 'Grape___healthy', 'Potato___Early_blight', 'Pepper,_bell___Bacterial_spot', 'Corn_(maize)___healthy', 'Apple___Apple_scab', 'Grape___Black_rot', 'Blueberry___healthy', 'Tomato___L

In [7]:
import shutil
for dir_path in [
    'kaggle_data/new-plant-diseases_dataset',
    'kaggle_data/new-plant-diseases_dataset/npd',
    'kaggle_data/new-plant-diseases_dataset/npd/npd-train/train',
    'kaggle_data/new-plant-diseases_dataset/npd/npd-train/valid',
    'kaggle_data/new-plant-diseases_dataset/test/test'
]:
    checkpoint_path = os.path.join(dir_path, '.ipynb_checkpoints')
    if os.path.exists(checkpoint_path):
        shutil.rmtree(checkpoint_path)

train_dir = 'kaggle_data/new-plant-diseases_dataset/npd'
test_dir_in_train = os.path.join(train_dir, 'test')
if os.path.exists(test_dir_in_train):
    shutil.rmtree(test_dir_in_train)
    print("Dossier 'test' supprimé de train.")

train_dir = 'kaggle_data/new-plant-diseases_dataset/npd/npd-train/train'
test_dir_in_train = os.path.join(train_dir, 'test')
if os.path.exists(test_dir_in_train):
    shutil.rmtree(test_dir_in_train)
    print("Dossier 'test' supprimé de train.")

In [8]:
# Removed redundant !pip install mlflow (already installed in the first cell)

### Model Training and Evaluation Pipeline

This is the core section of the notebook, implementing the training and validation pipeline for a plant disease classification model using EfficientNet. It covers:

*   **Device Configuration**: Sets up CUDA if available, otherwise uses CPU.
*   **Configuration**: Defines paths, batch size, epochs, and learning rate.
*   **Data Transforms**: Specifies image augmentations for training and basic resizing/normalization for validation.
*   **Dataset Loading**: Uses `ImageFolder` to load image datasets and `WeightedRandomSampler` to handle class imbalance during training.
*   **Model Initialization**: Loads a pre-trained EfficientNet-b0 model and replaces its final classification layer.
*   **Freezing Backbone**: Initially freezes the backbone layers and trains only the new classification head.
*   **Loss Function and Optimizer**: Uses CrossEntropyLoss with label smoothing and Adam optimizer.
*   **MLflow Integration**: Tracks training and validation metrics, best model, and confusion matrices.
*   **Fine-tuning**: Unfreezes later layers of the backbone and fine-tunes the model with a lower learning rate to improve performance.

In [9]:
import os
train_dir = 'kaggle_data/new-plant-diseases_dataset/test/test'
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        files = os.listdir(class_path)
        print(f"Classe {class_name} : {len(files)} fichiers")
        print(f"Exemples : {files[:3]}")

In [10]:
import os
from collections import Counter

train_dir = 'kaggle_data/new-plant-diseases_dataset/npd/npd-train/train'
valid_dir = 'kaggle_data/new-plant-diseases_dataset/npd/npd-train/valid'
test_dir = 'kaggle_data/new-plant-diseases_dataset/test/test'

def get_extensions(dir_path, name):
    extensions = []
    for root, _, files in os.walk(dir_path):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            extensions.append(ext)
    print(f"\nExtensions dans {name} :")
    print(Counter(extensions))

get_extensions(train_dir, 'train')
get_extensions(valid_dir, 'valid')
get_extensions(test_dir, 'test')


Extensions dans train :
Counter({'.jpg': 70295})

Extensions dans valid :
Counter({'.jpg': 17572})

Extensions dans test :
Counter({'.jpg': 33})


### Install MLflow

This cell installs MLflow, an open-source platform for managing the end-to-end machine learning lifecycle. It will be used to track experiments, log parameters, metrics, and models during the training process.

In [13]:
import os
import numpy as np
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

from efficientnet_pytorch import EfficientNet

# ==============================
# METRICS
# ==============================
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score
)

# ==============================
# MLflow
# ==============================
import mlflow
import mlflow.pytorch

# ==============================
# DEVICE
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# CONFIG
# ==============================
base_dir = "kaggle_data/new-plant-diseases_dataset/npd"
train_dir = os.path.join(base_dir, "npd-train/train")
valid_dir = os.path.join(base_dir, "npd-train/valid")

batch_size = 32
epochs = 15
lr = 1e-4

# ==============================
# TRANSFORMS
# ==============================
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

valid_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ==============================
# DATA
# ==============================
train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
valid_ds = datasets.ImageFolder(valid_dir, transform=valid_tf)

classes = train_ds.classes
num_classes = len(classes)

# ==============================
# WEIGHTED SAMPLER
# ==============================
class_counts = np.bincount(train_ds.targets)
weights = 1.0 / class_counts
sample_weights = weights[train_ds.targets]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)

# ==============================
# MODEL
# ==============================
model = EfficientNet.from_pretrained("efficientnet-b0")
model._fc = nn.Linear(model._fc.in_features, num_classes)
model = model.to(device)

# ==============================
# FREEZE BACKBONE (FIXED)
# ==============================
for name, param in model.named_parameters():
    if "_fc" not in name:
        param.requires_grad = False

# ==============================
# LOSS / OPTIM
# ==============================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model._fc.parameters(), lr=lr)

scaler = torch.amp.GradScaler()

# ==============================
# METRICS FUNCTIONS
# ==============================
def per_class_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, average=None)

def evaluate(y_true, y_pred):
    f1_macro = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=classes)
    return f1_macro, cm, report

# ==============================
# TRAIN LOOP (MLFLOW)
# ==============================
mlflow.set_experiment("plant_disease_research_pipeline")

best_val_loss = float("inf")

with mlflow.start_run():

    mlflow.log_param("model", "efficientnet-b0")
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("lr", lr)

    for epoch in range(epochs):

        # ================= TRAIN =================
        model.train()
        train_loss = 0
        train_preds, train_labels = [], []

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type='cuda'):
                out = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            train_preds.extend(out.argmax(1).cpu().numpy())
            train_labels.extend(y.cpu().numpy())

        train_loss /= len(train_loader)

        # ================= VALIDATION =================
        model.eval()
        val_loss = 0
        val_preds, val_labels = [], []

        with torch.no_grad():
            for x, y in valid_loader:
                x, y = x.to(device), y.to(device)

                with torch.amp.autocast(device_type='cuda'):
                    out = model(x)
                    loss = criterion(out, y)

                val_loss += loss.item()
                val_preds.extend(out.argmax(1).cpu().numpy())
                val_labels.extend(y.cpu().numpy())

        val_loss /= len(valid_loader)

        # ================= METRICS =================
        f1_macro, cm, report = evaluate(val_labels, val_preds)
        recalls = per_class_recall(val_labels, val_preds)

        val_acc = (np.array(val_preds) == np.array(val_labels)).mean()

        # ================= MLFLOW LOG =================
        mlflow.log_metric("train_loss", train_loss, epoch)
        mlflow.log_metric("val_loss", val_loss, epoch)
        mlflow.log_metric("val_acc", val_acc, epoch)
        mlflow.log_metric("f1_macro", f1_macro, epoch)

        for i, r in enumerate(recalls):
            mlflow.log_metric(f"recall_class_{i}", r, epoch)

        # ================= SAVE CONFUSION MATRIX =================
        np.save(f"cm_epoch_{epoch}.npy", cm)
        mlflow.log_artifact(f"cm_epoch_{epoch}.npy")

        print(f"\nEpoch {epoch+1}")
        print(f"Loss: {train_loss:.4f} | Val: {val_loss:.4f}")
        print(f"Acc: {val_acc:.4f} | F1: {f1_macro:.4f}")

        # ================= SAVE BEST =================
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(model.state_dict(), "best_model.pth")
            mlflow.pytorch.log_model(model, "model")

            print("✅ Best model saved")

# ==============================
# FINE TUNING (BACKBONE UNFREEZE)
# ==============================
print("\n🔓 Fine-tuning backbone...")

for name, p in model.named_parameters():
    if "blocks.6" in name or "blocks.5" in name:
        p.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

for epoch in range(5):
    model.train()
    loss_sum = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item()

    print(f"FT Epoch {epoch+1} Loss: {loss_sum/len(train_loader):.4f}")

torch.save(model.state_dict(), "final_model.pth")
mlflow.pytorch.log_model(model, "final_model")

print("🎯 FULL PIPELINE COMPLETED")

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 139MB/s] 
/tmp/ipykernel_3262/826970136.py:107: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler()


Loaded pretrained weights for efficientnet-b0


2026/05/04 12:18:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/04 12:18:31 INFO mlflow.store.db.utils: Updating database tables
2026/05/04 12:18:36 INFO mlflow.tracking.fluent: Experiment with name 'plant_disease_research_pipeline' does not exist. Creating a new experiment.
/tmp/ipykernel_3262/826970136.py:146: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast(device_type='cuda'):


KeyboardInterrupt: 

### Final Model Evaluation and Visualization

After the training and fine-tuning phases, this section loads the best-performing model (based on validation loss) and performs a comprehensive evaluation on the validation set. It generates and displays:

*   **Classification Report**: Provides precision, recall, F1-score, and support for each class, giving a detailed breakdown of model performance.
*   **Confusion Matrix**: Visualizes the performance of the classification model, showing the number of correct and incorrect predictions for each class. This helps identify specific classes where the model might be struggling.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model(model, dataloader, class_names):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    cm = confusion_matrix(all_labels, all_preds)

    report = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4
    )

    return cm, report

def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=False,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()


print("\n📊 Evaluation du meilleur modèle...")

# Charger le meilleur modèle
model.load_state_dict(torch.load("best_model.pth"))

cm, report = evaluate_model(model, valid_loader, train_ds.classes)

print("\n📌 Classification Report:")
print(report)

print("\n📌 Confusion Matrix:")
print(cm)

plot_confusion_matrix(cm, train_ds.classes)